In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import time
from datetime import datetime
import os

# Sklearn imports
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# Algoritmos de classificação
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, BaggingClassifier, VotingClassifier
)
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier, RidgeClassifier, 
    PassiveAggressiveClassifier, Perceptron
)
from sklearn.svm import SVC, NuSVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.neighbors import KNeighborsClassifier, RadiusNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, ComplementNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.gaussian_process import GaussianProcessClassifier

# XGBoost e LightGBM
try:
    from xgboost import XGBClassifier
    xgb_available = True
except ImportError:
    xgb_available = False

try:
    from lightgbm import LGBMClassifier
    lgb_available = True
except ImportError:
    lgb_available = False

try:
    from catboost import CatBoostClassifier
    catboost_available = True
except ImportError:
    catboost_available = False

print("🚀 EXPERIMENTO SISTEMÁTICO SKLEARN - TODOS OS ALGORITMOS")
print("=" * 70)

# Carregar dados
train_path = "/Users/marcelosilva/Desktop/projectOne/6/B- Binary Target DS DT/DatasetTrain.csv"
df_train = pd.read_csv(train_path)

print(f"✅ Dados carregados: {df_train.shape}")

# Verificar target
target_col = 'status_nutricional_who'
print(f"✅ Target distribution: {df_train[target_col].value_counts().to_dict()}")

# Limpar dados
exclude_cols = ['id_anon', 'vd_zimc']
df_clean = df_train.drop(columns=exclude_cols, errors='ignore')

# Conjuntos de features (baseado na análise SHAP)
feature_sets = {
    'top_1': ['imc_pre_gravidico'],
    'top_2': ['imc_pre_gravidico', 'k06_peso_engravidar'],
    'top_3': ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso'],
    'top_5': ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso', 'k07_peso_final', 'imc_final_gravidico'],
    'top_8': ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso', 'k07_peso_final', 
              'imc_final_gravidico', 'peso_concordancia', 'j03_cor', 'indice_ponderal'],
    'top_10': ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso', 'k07_peso_final', 
               'imc_final_gravidico', 'peso_concordancia', 'j03_cor', 'indice_ponderal',
               'delta_imc', 'ganho_relativo'],
    'antropometrico': ['imc_pre_gravidico', 'k06_peso_engravidar', 'k07_peso_final', 'h02_peso',
                      'imc_final_gravidico', 'h03_altura', 'indice_ponderal'],
    'demografico': ['j03_cor', 'd01_cor', 'b02_sexo', 'h04_parto'],
    'prenatal': ['k05_prenatal_consultas', 'k04_prenatal_semanas', 'cobertura_prenatal',
                'consultas_por_semana', 'adequacao_prenatal'],
    'core_mix': ['imc_pre_gravidico', 'j03_cor', 'k05_prenatal_consultas', 'h02_peso', 'h01_semanas_gravidez'],
    'all_features': [col for col in df_clean.columns if col != target_col]
}

# Definir algoritmos
algorithms = {
    # Ensemble Methods
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Bagging': BaggingClassifier(n_estimators=100, random_state=42),
    
    # Linear Models
    'LogisticRegression': LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    'SGD': SGDClassifier(class_weight='balanced', random_state=42, max_iter=1000),
    'Ridge': RidgeClassifier(class_weight='balanced', random_state=42),
    'PassiveAggressive': PassiveAggressiveClassifier(class_weight='balanced', random_state=42),
    
    # SVM
    'SVC_rbf': SVC(kernel='rbf', class_weight='balanced', random_state=42, probability=True),
    'SVC_linear': SVC(kernel='linear', class_weight='balanced', random_state=42, probability=True),
    'SVC_poly': SVC(kernel='poly', class_weight='balanced', random_state=42, probability=True),
    'LinearSVC': LinearSVC(class_weight='balanced', random_state=42, max_iter=2000),
    
    # Tree-based
    'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'ExtraTree': ExtraTreeClassifier(class_weight='balanced', random_state=42),
    
    # Neighbors
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'KNN_weighted': KNeighborsClassifier(n_neighbors=5, weights='distance'),
    
    # Naive Bayes
    'GaussianNB': GaussianNB(),
    'MultinomialNB': MultinomialNB(),
    'BernoulliNB': BernoulliNB(),
    'ComplementNB': ComplementNB(),
    
    # Discriminant Analysis
    'LDA': LinearDiscriminantAnalysis(),
    'QDA': QuadraticDiscriminantAnalysis(),
    
    # Neural Networks
    'MLP_small': MLPClassifier(hidden_layer_sizes=(100,), random_state=42, max_iter=500),
    'MLP_medium': MLPClassifier(hidden_layer_sizes=(100, 50), random_state=42, max_iter=500),
    
    # Gaussian Process
    'GaussianProcess': GaussianProcessClassifier(random_state=42),
}

# Adicionar algoritmos externos se disponíveis
if xgb_available:
    algorithms['XGBoost'] = XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0)

if lgb_available:
    algorithms['LightGBM'] = LGBMClassifier(random_state=42, verbosity=-1, force_row_wise=True)

if catboost_available:
    algorithms['CatBoost'] = CatBoostClassifier(random_state=42, verbose=False)

print(f"✅ Configurado {len(feature_sets)} conjuntos de features")
print(f"✅ Configurado {len(algorithms)} algoritmos")
print(f"✅ Total de experimentos: {len(feature_sets) * len(algorithms)}")

# Preprocessamento
def preprocess_data(df, features, target):
    """Preprocessa os dados para um conjunto específico de features"""
    
    # Verificar features disponíveis
    available_features = [f for f in features if f in df.columns]
    if len(available_features) != len(features):
        missing = set(features) - set(available_features)
        print(f"⚠️  Features não encontradas: {missing}")
    
    # Dataset com features + target
    df_subset = df[available_features + [target]].copy()
    
    # Separar features numéricas e categóricas
    numeric_features = df_subset.select_dtypes(include=[np.number]).columns.tolist()
    if target in numeric_features:
        numeric_features.remove(target)
    
    categorical_features = df_subset.select_dtypes(include=['object']).columns.tolist()
    
    # Tratar valores faltantes
    if len(numeric_features) > 0:
        imputer_num = SimpleImputer(strategy='median')
        df_subset[numeric_features] = imputer_num.fit_transform(df_subset[numeric_features])
    
    # Encoding de variáveis categóricas
    label_encoders = {}
    for col in categorical_features:
        le = LabelEncoder()
        df_subset[col] = df_subset[col].fillna('Missing')
        df_subset[col] = le.fit_transform(df_subset[col])
        label_encoders[col] = le
    
    X = df_subset.drop(columns=[target])
    y = df_subset[target]
    
    return X, y

# Função para avaliar modelo
def evaluate_model(model, X, y, cv_folds=5):
    """Avalia modelo com cross-validation"""
    
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    
    # Cross-validation scores
    cv_accuracy = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    cv_f1 = cross_val_score(model, X, y, cv=cv, scoring='f1')
    cv_precision = cross_val_score(model, X, y, cv=cv, scoring='precision')
    cv_recall = cross_val_score(model, X, y, cv=cv, scoring='recall')
    
    # AUC (só para modelos com predict_proba)
    try:
        cv_auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
        auc_mean = cv_auc.mean()
    except:
        auc_mean = 0
    
    results = {
        'accuracy_mean': cv_accuracy.mean(),
        'accuracy_std': cv_accuracy.std(),
        'f1_mean': cv_f1.mean(),
        'f1_std': cv_f1.std(),
        'precision_mean': cv_precision.mean(),
        'precision_std': cv_precision.std(),
        'recall_mean': cv_recall.mean(),
        'recall_std': cv_recall.std(),
        'auc_mean': auc_mean
    }
    
    return results

# Executar experimentos
results = []
experiment_count = 0
total_experiments = len(feature_sets) * len(algorithms)

print(f"\n🧪 INICIANDO {total_experiments} EXPERIMENTOS...")
print("=" * 70)

start_time = time.time()

for feature_set_name, features in feature_sets.items():
    print(f"\n📊 Testando conjunto: {feature_set_name} ({len(features)} features)")
    
    # Preprocessar dados para este conjunto
    try:
        X, y = preprocess_data(df_clean, features, target_col)
        
        # Testar cada algoritmo
        for alg_name, algorithm in algorithms.items():
            experiment_count += 1
            
            try:
                start_alg_time = time.time()
                
                # Avaliar modelo
                eval_results = evaluate_model(algorithm, X, y)
                
                end_alg_time = time.time()
                training_time = end_alg_time - start_alg_time
                
                # Armazenar resultado
                result = {
                    'experiment_id': experiment_count,
                    'feature_set': feature_set_name,
                    'n_features': len(features),
                    'algorithm': alg_name,
                    'training_time': training_time,
                    'features_used': ', '.join(features)
                }
                result.update(eval_results)
                results.append(result)
                
                print(f"✅ {experiment_count:3d}/{total_experiments} | {feature_set_name:12s} | {alg_name:15s} | Acc: {eval_results['accuracy_mean']:.3f} | F1: {eval_results['f1_mean']:.3f}")
                
            except Exception as e:
                print(f"❌ {experiment_count:3d}/{total_experiments} | {feature_set_name:12s} | {alg_name:15s} | ERRO: {str(e)[:50]}")
                
                result = {
                    'experiment_id': experiment_count,
                    'feature_set': feature_set_name,
                    'n_features': len(features),
                    'algorithm': alg_name,
                    'training_time': 0,
                    'features_used': ', '.join(features),
                    'accuracy_mean': 0,
                    'accuracy_std': 0,
                    'f1_mean': 0,
                    'f1_std': 0,
                    'precision_mean': 0,
                    'precision_std': 0,
                    'recall_mean': 0,
                    'recall_std': 0,
                    'auc_mean': 0,
                    'error': str(e)[:100]
                }
                results.append(result)
    
    except Exception as e:
        print(f"❌ Erro no preprocessamento para {feature_set_name}: {str(e)}")

# Análise dos resultados
results_df = pd.DataFrame(results)
total_time = time.time() - start_time

print(f"\n⏱️  EXPERIMENTOS CONCLUÍDOS em {total_time/60:.1f} minutos")
print("=" * 70)

# Filtrar sucessos
success_df = results_df[results_df['accuracy_mean'] > 0].copy()

print(f"\n📈 ANÁLISE DOS RESULTADOS:")
print(f"Total de experimentos realizados: {len(results_df)}")
print(f"Experimentos com sucesso: {len(success_df)}")

if len(success_df) > 0:
    # TOP 3 ALGORITMOS (por F1-Score médio)
    print(f"\n🏆 TOP 3 ALGORITMOS (por F1-Score médio):")
    top_algorithms = success_df.groupby('algorithm').agg({
        'accuracy_mean': 'mean',
        'f1_mean': 'mean',
        'auc_mean': 'mean',
        'training_time': 'mean'
    }).sort_values('f1_mean', ascending=False).head(3)
    
    for i, (alg, stats) in enumerate(top_algorithms.iterrows(), 1):
        print(f"{i}. {alg:20s} | F1: {stats['f1_mean']:.4f} | Acc: {stats['accuracy_mean']:.4f} | AUC: {stats['auc_mean']:.4f}")
    
    # TOP 3 CONJUNTOS DE FEATURES (por F1-Score médio)
    print(f"\n🎯 TOP 3 CONJUNTOS DE FEATURES (por F1-Score médio):")
    top_features = success_df.groupby('feature_set').agg({
        'accuracy_mean': 'mean',
        'f1_mean': 'mean',
        'auc_mean': 'mean',
        'n_features': 'first'
    }).sort_values('f1_mean', ascending=False).head(3)
    
    for i, (feat_set, stats) in enumerate(top_features.iterrows(), 1):
        print(f"{i}. {feat_set:15s} | F1: {stats['f1_mean']:.4f} | Acc: {stats['accuracy_mean']:.4f} | Features: {stats['n_features']}")
    
    # TOP 10 COMBINAÇÕES
    print(f"\n🎖️  TOP 10 MELHORES COMBINAÇÕES (por F1-Score):")
    top_combinations = success_df.nlargest(10, 'f1_mean')[['feature_set', 'algorithm', 'accuracy_mean', 'f1_mean', 'auc_mean', 'n_features']]
    
    for i, (_, row) in enumerate(top_combinations.iterrows(), 1):
        print(f"{i:2d}. {row['algorithm']:20s} + {row['feature_set']:12s} | F1: {row['f1_mean']:.4f} | Acc: {row['accuracy_mean']:.4f}")

    print(f"\n📊 RESUMO FINAL:")
    best_f1 = success_df['f1_mean'].max()
    best_acc = success_df['accuracy_mean'].max()
    baseline = 0.752  # Classe majoritária
    
    print(f"🎯 Melhor F1-Score: {best_f1:.4f}")
    print(f"🎯 Melhor Accuracy: {best_acc:.4f}")
    print(f"📈 Baseline (classe majoritária): {baseline:.3f}")
    print(f"🔥 Melhoria sobre baseline: {(best_acc - baseline)*100:+.1f}%")

# Salvar resultados
output_dir = "/Users/marcelosilva/Desktop/projectOne/6/E-Sklearn"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

os.makedirs(output_dir, exist_ok=True)

# Salvar todos os resultados
results_df.to_csv(f"{output_dir}/sklearn_systematic_results_{timestamp}.csv", index=False)

if len(success_df) > 0:
    success_df.to_csv(f"{output_dir}/sklearn_successful_experiments_{timestamp}.csv", index=False)
    top_algorithms.to_csv(f"{output_dir}/sklearn_top_3_algorithms_{timestamp}.csv")
    top_features.to_csv(f"{output_dir}/sklearn_top_3_feature_sets_{timestamp}.csv")
    top_combinations.to_csv(f"{output_dir}/sklearn_top_10_combinations_{timestamp}.csv", index=False)

print(f"\n💾 ARQUIVOS SALVOS:")
print(f"✅ sklearn_systematic_results_{timestamp}.csv - Todos os resultados")
print(f"✅ sklearn_successful_experiments_{timestamp}.csv - Sucessos")
print(f"✅ sklearn_top_3_algorithms_{timestamp}.csv - TOP 3 algoritmos")
print(f"✅ sklearn_top_3_feature_sets_{timestamp}.csv - TOP 3 feature sets")
print(f"✅ sklearn_top_10_combinations_{timestamp}.csv - TOP 10 combinações")

print(f"\n🎯 PRÓXIMOS PASSOS:")
print(f"1. Hyperparameter tuning dos TOP 3 algoritmos")
print(f"2. Ensemble dos melhores modelos")
print(f"3. Validação final no dataset de teste")
print(f"4. Análise de feature importance dos melhores")

print(f"\n🏁 EXPERIMENTO SKLEARN SISTEMÁTICO CONCLUÍDO!")